# Bypass vs. baseline

Does routing small flows on shorter paths cut their tail latency, what does it cost other traffic, and at what load does the cost outweigh the gain?

Reads the baseline and bypass runs of the sweep set by `SWEEP` in `common.py`. It only compares a setting and seed once the baseline and every bypass mode have finished it. The setting is the offered load, the hosts per router in `04-oversubscription`, or the small flows' share of bytes and the load in `06-permutation`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import common

common.setup_plots()
pd.options.display.float_format = "{:,.2f}".format

print(f"Reading {common.MACHINE}/{common.BATCH}")
baseline = common.load_runs(common.BASELINE_EXPERIMENT)
bypass = common.load_runs(common.BYPASS_EXPERIMENT)

# Keep only settings and seeds that the baseline and every bypass mode have finished, so
# that all modes in a table cover the same flows.
pairs = {(r.setting, r.seed) for r in baseline}
for mode in {r.bypass for r in bypass}:
    pairs &= {(r.setting, r.seed) for r in bypass if r.bypass == mode}
if not bypass:
    pairs = set()
runs = [r for r in baseline + bypass if (r.setting, r.seed) in pairs]
ready = bool(pairs)
if ready:
    flows = common.load_flows(runs)
    links = common.load_links(runs)
    print(f"Comparing {len(pairs)} setting and seed pairs: {sorted(pairs)}")
    modes = [m for m in common.MODES if m in {r.bypass for r in runs}]
    print(f"Modes: {modes}")
else:
    print(f"Nothing to compare yet: {len(baseline)} baseline runs and {len(bypass)} bypass runs share no setting and seed.")
    print(f"Run experiments/{common.BASELINE_EXPERIMENT}.sh and experiments/{common.BYPASS_EXPERIMENT}.sh.")

## Are the runs healthy and comparable?

Every mode must run the same flows, or the comparison means nothing.

In [ ]:
if ready:
    display(common.runs_table(runs))
    identity = ["flow", "source_router", "destination_router", "bytes", "start_ns"]
    mismatches = 0
    for key, group in flows.groupby([*common.SETTINGS, "seed"]):
        reference = group[group["bypass"] == "none"][identity].reset_index(drop=True)
        for mode, other in group.groupby("bypass"):
            if not reference.equals(other[identity].reset_index(drop=True)):
                print(f"Different flows at setting and seed {key}, mode {mode}")
                mismatches += 1
    print("All modes run the same flows" if mismatches == 0 else f"{mismatches} mismatches")

## H1: Does the bypass cut small-flow latency?

Expect the biggest gain for small flows between adjacent routers, since they cross one queue instead of two to five.

### Path lengths

Mean fabric hops of delivered packets, including ACKs. The adjacent bypass should take small flows between neighbors to 1 hop, and the shortest-path bypass should also shorten the paths of other small flows. Large flows should not change.

In [ ]:
if ready:
    display(common.mean_hops_table(runs))

In [ ]:
if ready:
    display(common.fct_table(flows, ["class", *common.SETTINGS, "bypass"]))

In [ ]:
if ready:
    common.plot_metric_by_mode(flows, "p50", "Median FCT (µs)")
    common.plot_metric_by_mode(flows, "p99", "p99 FCT (µs)")

### Flow by flow

Since every mode runs the same flows, each flow can be compared with itself. A ratio below 1 means the bypass made that flow faster.

In [ ]:
if ready:
    keys = [*common.SETTINGS, "seed", "flow"]
    reference = flows[flows["bypass"] == "none"][keys + ["fct_us"]].rename(columns={"fct_us": "baseline_us"})
    paired = flows[flows["bypass"] != "none"].merge(reference, on=keys)
    paired["ratio"] = paired["fct_us"] / paired["baseline_us"]
    display(
        paired.groupby(["class", *common.SETTINGS, "bypass"]).agg(
            median_ratio=("ratio", "median"),
            faster=("ratio", lambda r: (r < 1).mean()),
            slower=("ratio", lambda r: (r > 1).mean()),
        )
    )

### Why: fewer timeouts

A small flow's p99 is about 1,000 µs once more than 1% of its flows wait for a retransmission timeout, and much less otherwise: see the baseline notebook. So a bypass that brings that share under 1% cuts the p99 by a lot even if queueing barely changes. `any_timeout_pct` is the percentage that waited for any timeout, and `clean_p99` is the p99 of the rest, which is the queueing alone.

In [ ]:
if ready:
    display(common.timeout_table(flows, ["class", *common.SETTINGS, "bypass"]))

In [ ]:
if ready:
    common.plot_metric_by_mode(
        flows, "any_timeout_pct", "Flows with a timeout (%)", table=common.timeout_table, classes=common.SMALL_CLASSES
    )
    common.plot_metric_by_mode(
        flows, "clean_p99", "p99 FCT with no timeout (µs)", table=common.timeout_table, classes=common.SMALL_CLASSES
    )

### Seed by seed

Pooled numbers can hide a change that only some seeds show. Each row compares every seed's bypass run with the baseline run on the same flows: `seeds_fewer_timeouts` counts the seeds where the bypass cut the timeout rate, `timeout_pts` is the change in percentage points, and the `clean_` columns are percent changes. A change with the same sign in every seed is unlikely to be noise.

In [ ]:
if ready:
    display(common.seed_change_table(flows))

## H2: What does it cost, and where is the crossover?

Each ratio is a bypass mode's p99 over the baseline's at the same setting. Above 1 means the bypass made that class's tail worse. The crossover is the load, or the fabric load set by the hosts per router, where a small-flow ratio passes 1, and a large-flow ratio above 1 shows the cost to other traffic.

In [ ]:
if ready:
    p99 = common.fct_table(flows, ["class", *common.SETTINGS, "bypass"])["p99"].unstack("bypass")
    ratios = pd.DataFrame({mode: p99[mode] / p99["none"] for mode in p99.columns if mode != "none"})
    display(ratios)

### Large-flow throughput

The cost H2 expects: if bypassed small flows crowd the links large flows use, large-flow goodput drops, the slowest flows (`p1`) first. Ratios are each bypass mode over the baseline at the same setting.

In [ ]:
if ready:
    goodput = common.goodput_table(flows, [*common.SETTINGS, "bypass"])
    display(goodput)
    by_mode = goodput[["mean", "p1"]].unstack("bypass")
    display(pd.DataFrame({
        f"{stat}_{mode}": by_mode[(stat, mode)] / by_mode[(stat, "none")]
        for stat in ["mean", "p1"]
        for mode in by_mode[stat].columns if mode != "none"
    }))

In [ ]:
if ready:
    per_seed = common.goodput_table(flows, [*common.SETTINGS, "seed", "bypass"])[["mean", "p1"]]
    reference = per_seed.xs("none", level="bypass")
    changes = pd.concat(
        {mode: 100 * (per_seed.xs(mode, level="bypass") / reference - 1) for mode in modes if mode != "none"},
        names=["bypass"],
    ).add_suffix("_change_pct")
    display(changes.groupby(["bypass", *common.SETTINGS]).agg(["mean", "min", "max"]))

### Hotspots

The paper warns that shortest paths could congest direct links. `hotspot_ratio` is the busiest link over the mean link.

In [ ]:
if ready:
    table = common.link_table(links, [*common.SETTINGS, "bypass"])
    display(table)
    for title, rows in common.panels(table.reset_index()):
        figure, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
        for axis, column, label in zip(axes, ["max_utilization", "hotspot_ratio"], ["Busiest link's utilization", "Busiest link / mean link"]):
            for mode in common.MODES:
                group = rows[rows["bypass"] == mode]
                if group.empty:
                    continue
                axis.plot(group[common.SETTING], group[column], color=common.MODE_COLORS[mode], marker=common.MODE_MARKERS[mode], label=common.MODE_NAMES[mode])
            axis.set(ylabel=label, ylim=(0, None))
            common.label_setting_axis(axis)
        axes[0].legend()
        if title:
            figure.suptitle(title)
        figure.tight_layout()

### Which links got busier

The busiest link overall may not be the one the bypass loads, so this compares each link with itself in the baseline run. `links_busier` and `links_quieter` are the shares of links whose utilization moved by more than 1 point. The `top_` columns describe the 1% of links whose utilization rose the most. If the bypass creates hotspots, those links end up busier than the baseline's links and fill their queues more often than before.

In [ ]:
if ready:
    paired_links = common.link_changes(links)
    display(common.hotspot_table(paired_links, [*common.SETTINGS, "bypass"]))

In [ ]:
if ready:
    # The most loaded setting of each panel, where a hotspot is likeliest.
    for title, rows in common.panels(paired_links):
        top = rows[common.SETTING].max()
        figure, axis = plt.subplots(figsize=(6, 3.4))
        for mode, group in rows[rows[common.SETTING] == top].groupby("bypass"):
            change = np.sort(group["utilization_change"].to_numpy()) * 100
            axis.plot(change, np.linspace(0, 1, len(change)), color=common.MODE_COLORS[mode], label=common.MODE_NAMES[mode])
        axis.axvline(0, color="#b5b4ad", linewidth=1)
        name = common.setting_name(top)
        axis.set(xlabel="Change in link utilization (points)", ylabel="Share of links at or below", title=f"{title}, {name}" if title else name)
        axis.legend()
        figure.tight_layout()

### Fan-in

Does the bypass concentrate a destination's traffic on fewer neighbors?

In [ ]:
if ready:
    display(common.fan_in_table(links, [*common.SETTINGS, "bypass"]))

## H3: How many flows does it reach?

An adjacent-only bypass can only help small flows between neighboring routers. Comparing all small flows shows what that limit costs against bypassing everything.

In [ ]:
if ready:
    small = flows[flows["class"] != "large"]
    print(f"Adjacent share of small flows: {(small['class'] == 'small_adjacent').mean():.1%}")
    display(common.fct_table(small.assign(group="all small"), ["group", *common.SETTINGS, "bypass"]))